## llama.cpp GGUF backend — Colab T4 1x

Fill parameters and run.

In [ ]:
#@title llama.cpp GGUF backend
REPO_URL = "https://github.com/N3iKos/llama-cpp-notebook" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
MODEL_URL = "https://huggingface.co/ggml-org/Qwen2.5-VL-3B-Instruct-GGUF/resolve/main/Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf" #@param {type:"string"}
MMPROJ_URL = "https://huggingface.co/ggml-org/Qwen2.5-VL-3B-Instruct-GGUF/resolve/main/mmproj-Qwen2.5-VL-3B-Instruct-Q8_0.gguf" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}
NGROK_AUTHTOKEN = "" #@param {type:"string"}
TUNNEL_MODE = "both" #@param ["both", "ngrok", "cloudflare"]
CTX_SIZE = 8192 #@param {type:"integer"}
BATCH_SIZE = 2048 #@param {type:"integer"}
UBATCH_SIZE = 512 #@param {type:"integer"}
PARALLEL = 1 #@param {type:"integer"}
ENABLE_FLASH_ATTN = True #@param {type:"boolean"}
ENABLE_MMPROJ_OFFLOAD = True #@param {type:"boolean"}
IMAGE_MIN_TOKENS = "" #@param {type:"string"}
IMAGE_MAX_TOKENS = "" #@param {type:"string"}
CHAT_TEMPLATE_KWARGS = "" #@param {type:"string"}

import sys
import subprocess
import os
import time
import html
from collections import deque
from pathlib import Path
from IPython.display import HTML, display

def _bootstrap_panel_html(label, command, status, elapsed, exit_code, log_path, lines):
    badge = status.lower()
    escaped_lines = "\n".join(html.escape(x) for x in lines)
    exit_text = "exit -" if exit_code is None else f"exit {exit_code}"
    return f"""
<style>
.ios-terminal-panel{{overflow:hidden;margin:12px 0;border:1px solid rgba(255,255,255,.10);border-radius:22px;background:linear-gradient(145deg,rgba(42,47,58,.92),rgba(16,18,24,.94));box-shadow:0 18px 48px rgba(0,0,0,.28),inset 0 1px 0 rgba(255,255,255,.08);color:#e8edf7;font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace}}
.ios-terminal-titlebar{{display:flex;align-items:center;gap:10px;padding:12px 16px;background:rgba(255,255,255,.055);border-bottom:1px solid rgba(255,255,255,.08)}}
.ios-terminal-lights{{display:flex;gap:7px}}.traffic-light{{width:12px;height:12px;border-radius:999px;box-shadow:inset 0 0 0 1px rgba(0,0,0,.16)}}.red{{background:#ff5f57}}.yellow{{background:#febc2e}}.green{{background:#28c840}}
.ios-terminal-title{{flex:1;min-width:0;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;color:#f6f8fc;font-size:13px;font-weight:700;letter-spacing:0}}
.ios-terminal-badge{{border-radius:999px;padding:4px 9px;font-size:11px;font-weight:800;letter-spacing:0}}.ios-terminal-badge.running{{color:#c7f7ff;background:rgba(65,196,255,.16)}}.ios-terminal-badge.done{{color:#c9ffd8;background:rgba(40,200,64,.18)}}.ios-terminal-badge.failed{{color:#ffd0d0;background:rgba(255,95,87,.20)}}
.ios-terminal-meta{{display:grid;grid-template-columns:repeat(3,max-content) minmax(0,1fr);gap:10px;padding:10px 16px 0;color:#aeb8c8;font-size:12px}}.ios-terminal-meta span{{min-width:0;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}}
.ios-terminal-command{{padding:8px 16px 0;color:#d7e0ef;font-size:12px;white-space:pre-wrap;overflow-wrap:anywhere}}.ios-terminal-output{{margin:10px 0 0;padding:0 16px 16px;max-height:460px;overflow:auto;color:#e9eef8;font-size:12px;line-height:1.45;white-space:pre-wrap}}
</style>
<div class='ios-terminal-panel'><div class='ios-terminal-titlebar'><div class='ios-terminal-lights'><span class='traffic-light red'></span><span class='traffic-light yellow'></span><span class='traffic-light green'></span></div><div class='ios-terminal-title'>{html.escape(label)}</div><span class='ios-terminal-badge {badge}'>{html.escape(status)}</span></div><div class='ios-terminal-meta'><span>{elapsed:.1f}s</span><span>{html.escape(exit_text)}</span><span>log</span><span title='{html.escape(str(log_path))}'>{html.escape(str(log_path))}</span></div><div class='ios-terminal-command'>$ {html.escape(command)}</div><pre class='ios-terminal-output'>{escaped_lines}</pre></div>
"""

def run_bootstrap(cmd, label, log_path, tail_lines=40, failure_tail_lines=30):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    command = " ".join(map(str, cmd))
    lines = deque(maxlen=tail_lines)
    start = time.time()
    handle = display(HTML(_bootstrap_panel_html(label, command, "RUNNING", 0.0, None, log_path, [])), display_id=True)
    last_update = 0.0
    proc = None
    try:
        import pty, select
        master_fd, slave_fd = pty.openpty()
        proc = subprocess.Popen(cmd, stdin=slave_fd, stdout=slave_fd, stderr=slave_fd, close_fds=True)
        os.close(slave_fd)
        with log_path.open("w", encoding="utf-8", errors="replace") as log:
            while True:
                readable, _, _ = select.select([master_fd], [], [], 0.05)
                if readable:
                    try:
                        raw = os.read(master_fd, 4096)
                    except OSError:
                        raw = b""
                    chunk = raw.decode("utf-8", "replace")
                    log.write(chunk); log.flush()
                    for line in chunk.splitlines(): lines.append(line.rstrip("\r"))
                now = time.time()
                if now - last_update >= 0.08:
                    handle.update(HTML(_bootstrap_panel_html(label, command, "RUNNING", now - start, None, log_path, list(lines))))
                    last_update = now
                if proc.poll() is not None:
                    break
        try:
            os.close(master_fd)
        except OSError:
            pass
    except Exception:
        if proc is not None:
            raise
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        with log_path.open("w", encoding="utf-8", errors="replace") as log:
            for line in proc.stdout:
                log.write(line); log.flush(); lines.append(line.rstrip("\n"))
                now = time.time()
                if now - last_update >= 0.08:
                    handle.update(HTML(_bootstrap_panel_html(label, command, "RUNNING", now - start, None, log_path, list(lines))))
                    last_update = now
            proc.wait()
    status = "DONE" if proc.returncode == 0 else "FAILED"
    shown = list(lines)[-tail_lines if proc.returncode == 0 else -failure_tail_lines:]
    handle.update(HTML(_bootstrap_panel_html(label, command, status, time.time() - start, proc.returncode, log_path, shown)))
    if proc.returncode != 0:
        raise RuntimeError(f"{label} failed with exit code {proc.returncode}. log: {log_path}")

repo_dir = Path("/content/llama-cpp-notebook")
if not repo_dir.exists():
    run_bootstrap(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)], "clone repository", "/content/llama_cpp_notebook_clone.log")
else:
    run_bootstrap(["git", "-C", str(repo_dir), "pull"], "update repository", "/content/llama_cpp_notebook_pull.log")

run_bootstrap([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], "install package", "/content/llama_cpp_notebook_pip_install.log")
sys.path.insert(0, str(repo_dir))

def optional_int(value):
    value = str(value).strip()
    return int(value) if value else None

from gguf_backend.colab_runner import run_colab

result = run_colab(
    model_url=MODEL_URL,
    mmproj_url=MMPROJ_URL,
    hf_token=HF_TOKEN,
    ngrok_authtoken=NGROK_AUTHTOKEN,
    tunnel_mode=TUNNEL_MODE,
    ctx_size=CTX_SIZE,
    split_mode="none",
    tensor_split="1",
    batch_size=BATCH_SIZE,
    ubatch_size=UBATCH_SIZE,
    parallel=PARALLEL,
    flash_attn=ENABLE_FLASH_ATTN,
    image_min_tokens=optional_int(IMAGE_MIN_TOKENS),
    image_max_tokens=optional_int(IMAGE_MAX_TOKENS),
    chat_template_kwargs=CHAT_TEMPLATE_KWARGS.strip() or None,
    mmproj_offload=ENABLE_MMPROJ_OFFLOAD,
    port=8080,
    alias="local-vl",
)

print(result)